In [30]:
import pandas as pd
df = pd.read_json('/workspaces/Edu_Math_tutor/single_error.jsonl', lines = True)
#df.to_json('single_error.jsonl', force_ascii= False, orient= 'records', lines=True)

In [31]:
df

,topic,question,correct_answer,full_answer,wrong_solution,explanation,error_type,bloom_level
0,"Hoán vị, tổ hợp, chỉnh hợp",Có 5 nam sinh và 3 nữ sinh cần được xếp vào mộ...,None,None,Xét câu b) Xếp học sinh cùng giới đứng cạnh nh...,Lời giải đã bỏ qua một bước quan trọng. Sau kh...,[MS],3.0
1,"Hoán vị, tổ hợp, chỉnh hợp",Có 5 nam sinh và 3 nữ sinh cần được xếp vào mộ...,None,None,Xét câu d) Xếp học sinh nam luôn đứng cạnh nha...,Học sinh đã xác định đúng các bước cần làm như...,[OP],3.0
2,"Hoán vị, tổ hợp, chỉnh hợp",Một trường THPT X có 8 giáo viên Toán gồm có 3...,None,None,Xét câu c) Chọn 1 giáo viên nam môn Toán và 1 ...,Việc chọn một giáo viên nam môn Toán và một gi...,[OP],3.0
3,"Hoán vị, tổ hợp, chỉnh hợp",Một trường THPT X có 8 giáo viên Toán gồm có 3...,None,None,Xét câu d) Chọn đoàn 3 người có đủ 2 môn và đủ...,Lời giải đã thiếu một trường hợp quan trọng là...,[MS],3.0
4,"Hoán vị, tổ hợp, chỉnh hợp","Có 5 bông hồng, 4 bông trắng (mỗi bông đều khá...",None,None,Xét câu b) Số cách chọn 4 bông mà số bông mỗi ...,"Lập luận của bài giải là đúng, tuy nhiên đã có...",[CAL],3.0
...,...,...,...,...,...,...,...,...
1025,Vị trí tương đối trong mp toạ độ,"Câu 5: Trong mặt phẳng tọa độ Oxy, cho điểm A(...",None,None,Đường thẳng Δ cách đều hai điểm A và M khi và ...,Lý luận sai (REAS) khi học sinh chỉ xét một tr...,"[REAS, CAL]",NaN
1026,Vị trí tương đối trong mp toạ độ,Câu 1: Viết phương trình tổng quát của đường t...,None,None,"Đường thẳng Δ có VTPT là $\vec{n}_{\Delta}=(3,...",Học sinh đã áp dụng sai công thức (FC) khi cho...,"[FC, CAL]",NaN
1027,Vị trí tương đối trong mp toạ độ,Câu 1: Viết phương trình tổng quát của đường t...,None,None,"Đường thẳng Δ có VTPT là $\vec{n}_{\Delta}=(3,...",Học sinh có lý luận (REAS) đúng về mối quan hệ...,"[FC, REAS]",NaN
1028,Vị trí tương đối trong mp toạ độ,Câu 2: Tìm giá trị thực của tham số m để ba đư...,None,None,"Để ba đường thẳng đồng quy, chúng phải cùng đi...",Học sinh có hướng lý luận (REAS) đúng là tìm g...,"[CAL, REAS]",NaN


In [1]:
import pandas as pd
import json

# Load JSON file
with open("/workspaces/Edu_Math_tutor/wrong_answer/question_1.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Step 1: Go inside wrong_solutions, keep meta
df = pd.json_normalize(
    data,
    record_path=["wrong_solutions"],            # dive into wrong_solutions
    meta=["question"]                           # keep question for context
)

# Step 2: Explode applied_errors (list of dicts)
df = df.explode("applied_errors")

# Step 3: Flatten applied_errors into separate columns
df = pd.concat(
    [df.drop(columns=["applied_errors"]),
     df["applied_errors"].apply(pd.Series)], 
    axis=1
)

# Reorder columns for clarity
df = df[[
    "question", 
    "transformed_solution", 
    "wrong_step", 
    "error_type", 
    "description", 
    "is_single_error", 
    "notes"
]]

print(df)


                                            question  \
0  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
1  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
2  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
3  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
4  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
5  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   
6  Xét phép thử "Gieo một xúc sắc cân đối và đồng...   

                                transformed_solution  wrong_step error_type  \
0  Xúc sắc có 6 mặt, các mặt có số chấm từ 1 đến ...           5        MIS   
1  Xúc sắc có 6 mặt. \nA. Mặt xuất hiện có số chấ...           5        MIS   
2  Xúc sắc có 6 mặt, các mặt có số chấm từ 1 đến ...           1         HA   
3  Xúc sắc có 6 mặt. \nA. Mặt xuất hiện có số chấ...           5       REAS   
4  Xúc sắc có 6 mặt. \nA. Mặt xuất hiện có số chấ...           5         HA   
5  Xúc sắc có 6 mặt. \nA. Mặt xuất hiện có số chấ...           5        MIS  

In [6]:
questions = df['question'].unique()

In [8]:
questions_df = pd.DataFrame(questions, columns=['question'])

In [11]:
questions_df.to_csv("unique_questions.csv", index=False)

In [27]:
prompt_template = """You are given a math problem as input.
{question}
Your task is to generate several wrong answers in the style of a student’s handwritten solution. The wrong answers should feel natural and realistic, including errors such as: misunderstanding the problem, misusing formulas, calculation mistakes, or flawed reasoning.

When you write the wrong solution:

Show the step-by-step solution as if a student is solving it.

Intentionally insert multiple errors of forms (conceptual, logical, arithmetic, comprehension, etc.).

Make the mistakes look plausible, not random—like what a student would naturally do.

After writing the wrong solution, list all the errors you made and classify each into one of these categories:

🔹 Error Taxonomy

1. Lỗi Tính toán (Calculation Errors)

CAL: Lỗi tính toán số học (Arithmetic miscalculation)

UC: Sai khi đổi/nhầm đơn vị (Unit conversion mistake)

OP: Sử dụng toán tử không đúng (Operator misuse)

CO: Lỗi đếm, sai số lượng hoặc chỉ số (Counting/index mistake)

2. Lỗi Khái niệm (Conceptual Errors)

FC: Nhầm lẫn hoặc áp dụng sai công thức (Formula confusion)

KNOW: Nhớ hoặc hiểu sai kiến thức (Knowledge misunderstanding)

CV: Gán sai thuộc tính giá trị (Wrong value attribution)

3. Lỗi Logic (Logical Errors)

REAS: Lý luận sai, suy luận không đúng (Faulty reasoning)

MS: Thiếu bước trung gian quan trọng (Missing step)

CS: Các bước mâu thuẫn lẫn nhau (Contradiction in steps)

HA: Thêm thông tin bịa/không liên quan (Hallucinated info)

4. Lỗi Đọc hiểu (Comprehension Errors)

MIS: Hiểu sai yêu cầu đề bài (Misunderstood the question)

📌 Output Format

The output must be the original JSON input plus an additional field:

{{
  "question": "...",
  "wrong_answers": [
    {{
      "error_solution": "Step-by-step wrong solution here...",
      "error_type": ["CAL", "KNOW"],
      "explanation":"....."
}},
    {{
      "error_solution": "Another wrong solution with different mistakes...",
      "error_type": ["MIS", "FC"],
      "explanation":"...."
    }}
  ]
}}
"error_solution" = the full wrong solution written as if by a student.

"error_type" = list of all error codes that appear in that solution.
"explanation" = explanation for the error_solution
"""

In [28]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import pandas as pd
import dotenv
import os
dotenv.load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API")

df = pd.read_json('selected_questions.json')
reasoning_model = "models/gemma-3-27b-it"
#llm_reasoning = ChatGoogleGenerativeAI(model=reasoning_model, temperature=0.2, top_p=0.9)
prompt = PromptTemplate.from_template(prompt_template)
import json
import re

def generate_wrong_solutions(llm_reasoning, question, file_path="wrong_solutions.jsonl"):
    chain = prompt | llm_reasoning
    raw_output = chain.invoke({"question": question}).content

    # Step 1: Extract JSON part with regex
    match = re.search(r"\{.*\}", raw_output, re.DOTALL)
    if match:
        json_text = match.group(0)
    else:
        # fallback if no JSON found
        json_text = None

    # Step 2: Parse into Python dict
    try:
        result_json = json.loads(json_text)
    except json.JSONDecodeError:
        result_json = None

    # Step 3: Append to JSONL file
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result_json, ensure_ascii=False) + "\n")

    # Optional: return if you want to use it in code
    #return result_json



In [34]:
import random
for i in range(2):
  temp = random.uniform(0.2, 0.7)   # pick random temp in [0.2, 0.7]
  top_p = random.uniform(0.7, 0.9)  # pick random top_p in [0.7, 0.9]
  llm = ChatGoogleGenerativeAI(
        model=reasoning_model,
        temperature=temp,
        top_p=top_p,
    )
  for index, row in df.iterrows():
      print(row)
      generate_wrong_solutions(llm, row['question'])
      break

topic                           Hệ bất phương trình bậc nhất hai ẩn
question          Cho hệ bất phương trình { 2y - 2x ≤ 2; 2y - x ...
correct_answer                                                 None
full_answer                                                    None
Name: 0, dtype: object
topic                           Hệ bất phương trình bậc nhất hai ẩn
question          Cho hệ bất phương trình { 2y - 2x ≤ 2; 2y - x ...
correct_answer                                                 None
full_answer                                                    None
Name: 0, dtype: object


In [ ]:
import os
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import glob, re
import pandas as pd
# --- Environment Setup ---
import dotenv
dotenv.load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API")

DEFAULT_REASONING_MODEL = "models/gemma-3-27b-it"
#models/gemini-2.5-flash
DEFAULT_FAST_MODEL = "models/gemma-3-27b-it"
wrong_answer_dir = "//workspaces//Edu_Math_tutor//test"

def natural_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

csv_files = sorted(glob.glob(os.path.join(wrong_answer_dir, "student_error_*.csv")), key=natural_key)
prompt_template ="""Evaluate the error_analysis {error_analysis} against the groud truth explanation {explanation}.
                        You return a json structure:
                        point: 1 if error_analysis match ground truth, 0 if not.
                        reason: explain why.
                     """
llm = ChatGoogleGenerativeAI(model=DEFAULT_REASONING_MODEL, temperature=0.2)
prompt = PromptTemplate.from_template(prompt_template)
chain = prompt | llm
for file in csv_files:
    df = pd.read_csv(file)
    df['res'] = None
    for index, row in df.iterrows():
        res = chain.invoke({"error_analysis": row['agent_analysis'], "explanation": row['explanation']})
        df.at[index, 'res'] = res
    df.to_csv(file)

In [15]:
df = pd.read_csv(csv_files[2])
df

,Unnamed: 0.1,Unnamed: 0,topic,question,correct_answer,full_answer,wrong_solution,explanation,Errortype,bloom_level,agent_analysis,res
0,0,0,"Hoán vị, tổ hợp, chỉnh hợp",Có 5 nam sinh và 3 nữ sinh cần được xếp vào mộ...,NaN,NaN,Xét câu b) Xếp học sinh cùng giới đứng cạnh nh...,Lời giải đã bỏ qua một bước quan trọng. Sau kh...,Missing Step (MS),3,detailed_analysis='Học sinh mắc lỗi trong việc...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
1,1,1,"Hoán vị, tổ hợp, chỉnh hợp",Có 5 nam sinh và 3 nữ sinh cần được xếp vào mộ...,NaN,NaN,Xét câu d) Xếp học sinh nam luôn đứng cạnh nha...,Học sinh đã xác định đúng các bước cần làm như...,Operator Error (OP),3,detailed_analysis='Học sinh mắc lỗi trong việc...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
2,2,2,"Hoán vị, tổ hợp, chỉnh hợp",Một trường THPT X có 8 giáo viên Toán gồm có 3...,NaN,NaN,Xét câu c) Chọn 1 giáo viên nam môn Toán và 1 ...,Việc chọn một giáo viên nam môn Toán và một gi...,Operator Error (OP),3,detailed_analysis='Phân tích từ các detector c...,"content='```json\n{\n ""point"": 0,\n ""reason""..."
3,3,3,"Hoán vị, tổ hợp, chỉnh hợp",Một trường THPT X có 8 giáo viên Toán gồm có 3...,NaN,NaN,Xét câu d) Chọn đoàn 3 người có đủ 2 môn và đủ...,Lời giải đã thiếu một trường hợp quan trọng là...,Missing Step (MS),3,detailed_analysis='Phân tích các lỗi cho thấy ...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
4,4,4,"Hoán vị, tổ hợp, chỉnh hợp","Có 5 bông hồng, 4 bông trắng (mỗi bông đều khá...",NaN,NaN,Xét câu b) Số cách chọn 4 bông mà số bông mỗi ...,"Lập luận của bài giải là đúng, tuy nhiên đã có...",Calculation Error (CAL),3,detailed_analysis='Phân tích từ các detector c...,"content='```json\n{\n ""point"": 0,\n ""reason""..."
5,5,5,"Hoán vị, tổ hợp, chỉnh hợp","Có 5 bông hồng, 4 bông trắng (mỗi bông đều khá...",NaN,NaN,Xét câu d) Số cách chọn 4 bông có đủ hai màu:\...,Phương pháp giải này dẫn đến việc đếm lặp các ...,Reasoning Error (REAS),3,"detailed_analysis=""Bài giải câu d) mắc phải lỗ...","content='```json\n{\n ""point"": 1,\n ""reason""..."
6,6,6,"Hoán vị, tổ hợp, chỉnh hợp",Cho hai đường thẳng song song d₁ và d₂. Trên d...,NaN,NaN,Tổng số điểm là 17 + 20 = 37 điểm.\nĐể tạo thà...,Lời giải đã không xét đến điều kiện để tạo thà...,Knowledge Error (KNOW),3,detailed_analysis='Bài giải mắc các lỗi về mặt...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
7,7,7,"Hoán vị, tổ hợp, chỉnh hợp",Cho hai đường thẳng song song d₁ và d₂. Trên d...,NaN,NaN,"Để tạo thành một tam giác, các đỉnh không được...",Học sinh đã xác định đúng số lượng các bộ điểm...,Misinterpretation of the Question (MIS),3,detailed_analysis='Bài giải thể hiện sự nhầm l...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
8,8,8,"Hoán vị, tổ hợp, chỉnh hợp","Cho 10 điểm phân biệt A₁, A₂,..., A₁₀ trong đó...",NaN,NaN,Số cách chọn 3 điểm bất kỳ từ 10 điểm đã cho l...,Lời giải đã bỏ qua bước quan trọng nhất là loạ...,Missing Step (MS),3,detailed_analysis='Học sinh gặp vấn đề trong v...,"content='```json\n{\n ""point"": 1,\n ""reason""..."
9,9,9,"Hoán vị, tổ hợp, chỉnh hợp","Cho 10 điểm phân biệt A₁, A₂,..., A₁₀ trong đó...",NaN,NaN,Ta sẽ đếm trực tiếp số tam giác.\nCác điểm đượ...,"Phương pháp đếm trực tiếp là hợp lệ, nhưng lời...",Missing Step (MS),3,detailed_analysis='Bài giải của học sinh mắc l...,"content='```json\n{\n ""point"": 1,\n ""reason""..."


file 10: 8/10
file 20: 7/10